### LangGraph核心组件：时光旅行

- 时光旅行允许用户回溯和修改过去的状态
- 重放是对过去状态的重播
- 分叉则允许重新访问智能体的过去操作并探索图中的替代路径

In [34]:
# 引入工具
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import MessagesState, START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# 创建一个工具
@tool
def play_song_qq(song: str):
    """播放QQ音乐上的歌曲"""
    # 调用QQ音乐的API播放歌曲
    return f"正在播放QQ音乐上的歌曲: {song}"

@tool
def play_song_on_163(song: str):
    """播放网易云音乐上的歌曲"""
    # 调用网易云音乐的API播放歌曲
    return f"正在播放网易云音乐上的歌曲: {song}"

tools = [play_song_qq, play_song_on_163]
tool_node = ToolNode(tools)

# 设置模型
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

openai = ChatOpenAI(
    model="Qwen/Qwen3.8-27B", 
    temperature=0,
    api_key= os.getenv("api_key"),
    base_url=os.getenv("base_url"),
)
 
# 绑定工具 parallel_tool_calls=False 表示不并行调用工具
model = openai.bind_tools(tools, parallel_tool_calls=False)

# 定义节点和条件边

# 定义确定是否继续的函数
def should_continue(state):
    messages = state["messages"]
    last_message = messages[-1]
    # 如果最后一条消息没有工具调用，则结束
    if not last_message.tool_calls:
        return "end"
    # 如果最后一条消息有工具调用，则继续
    else:
        return "continue"

# 定义调用模型的函数
def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    # 将模型的响应添加到消息列表中
    return {"messages": [response]}

# 定义状态图
workflow = StateGraph(MessagesState)

# 定义状态图的节点和边
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 将入口点设置为 "agent"
# 这意味着这个节点是第一个被调用的
workflow.add_edge(START, "agent")

# 现在添加一个条件边
workflow.add_conditional_edges(
    # 首先，定义起始节点。使用`agent`
    # 这意味着这些是在调用`agent`节点后采取的边
    "agent", 
    # 确定下一个调用哪个节点的函数
    should_continue,
    # 最后传入一个映射，键是字符串，值是其他节点
    # END是一个特殊的节点，表示状态图的结束
    # 调用`should_continue`函数后，如果返回"continue"，则调用"action"节点
    # 如果返回"end"，则调用END节点
    {
        # 如果是"tools", 则调用工具节点
        "continue": "action",
        # 否则结束
        "end": END,
    }
)

# 现在从 tools 到 agent 添加一个普通边。
# 这意味着在调用 tools 之后，下一步调用 agent节点。
workflow.add_edge("action", "agent")

# 设置内存
memory = MemorySaver()

# 最后编译
# 将它编译成一个LangChain Runnable对象
# 意味着你可以像使用任何其他LangChain Runnable对象一样使用它

# 添加'interrupt_before=["action"]'参数，表示在调用'action'节点之前中断

app = workflow.compile(checkpointer=memory)

进行简单交互，要求播放周董的歌曲

查看记录并重放

In [33]:
app.get_state(config).values["messages"]

KeyError: 'messages'

In [23]:
all_states = []
for state in app.get_state_history(config):
    print(state)
    all_states.append(state)
    print("-----------")

我们可以返回任何一个状态节点，并从那个时候重新开始操作

In [26]:
to_replay = all_states[2]
to_replay.values

IndexError: list index out of range

In [27]:
# 查看下一个节点
to_replay.next

NameError: name 'to_replay' is not defined

如果想从这个状态节点重播，只需这样

In [28]:
for event in app.stream(None, to_replay.config):
    for v in event.values():
        print(v)

NameError: name 'to_replay' is not defined

#### 分叉操作
***
从某个节点开始操作，对执行数据进行分叉

In [29]:
# 修改最后一个消息的工具调用
# 将其从 `play_song_on_qq` 更改为 `paly_song_on_163`
last_message = to_replay.values["messages"][-1]
last_message.tool_calls[0]["name"] = "play_song_on_163"

branch_config = app.update_state(
    to_replay.config,
    {"messages": [last_message]}
)

NameError: name 'to_replay' is not defined

此时整个图的流就进行了分叉处理

In [19]:
for event in app.stream(None, branch_config):
    for v in event.values():
        print(v)

{'messages': [ToolMessage(content='正在播放网易云音乐上的歌曲: 夜曲', name='play_song_on_163', id='2d15bb72-30f2-4b9d-bcca-6c6d5884d91a', tool_call_id='call_75034a501d284e19bedcc1')]}
{'messages': [AIMessage(content='好的，正在播放网易云音乐上的歌曲《夜曲》。希望您喜欢！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 291, 'total_tokens': 309, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-f96ba9ba-70b9-9c4d-8bc5-118ea381441e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06000-9478-72a3-91a7-42bb5cabdd32-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 291, 'output_tokens': 18, 'total_tokens': 309, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}
